# Find and tag notes with no cloze syntax (AnkiConnect)

This notebook connects to Anki via the AnkiConnect plugin, checks the `Cloze` field, and tags notes that do not contain cloze syntax like `{{c1::...}}`.

Run once with `DRY_RUN = True`, then set `DRY_RUN = False` to apply tags.

In [4]:
import json
import re
import urllib.request

ANKI_CONNECT_URL = "http://127.0.0.1:8765"
ANKI_CONNECT_VERSION = 6

def invoke(action, **params):
    payload = json.dumps({"action": action, "version": ANKI_CONNECT_VERSION, "params": params}).encode("utf-8")
    req = urllib.request.Request(ANKI_CONNECT_URL, data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    if data.get("error"):
        raise RuntimeError(f"AnkiConnect error in {action}: {data['error']}")
    return data.get("result")

print("Connected AnkiConnect version:", invoke("version"))

Connected AnkiConnect version: 6


In [5]:
# Configuration
QUERY = 'note:"Moritz Language Reactor"'  # e.g. 'note:"My Cloze Type"' or 'deck:"Japanese"'
FIELD_NAME = 'Cloze'
TAG = 'no_cloze_syntax'
DRY_RUN = False

# Matches {{c1::text}} and {{c1::text::hint}}
CLOZE_RE = re.compile(r'\{\{c\d+::.*?\}\}', re.DOTALL)

In [6]:
def chunked(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]

nids = invoke('findNotes', query=QUERY)
print(f'Found notes: {len(nids)}')

to_tag = []
checked = 0
missing_field = 0
preview = []

for batch in chunked(nids, 500):
    infos = invoke('notesInfo', notes=batch)
    for info in infos:
        fields = info.get('fields', {})
        fobj = fields.get(FIELD_NAME)

        if fobj is None:
            missing_field += 1
            continue

        checked += 1
        value = (fobj.get('value', '') or '')

        if not CLOZE_RE.search(value):
            nid = info['noteId']
            to_tag.append(nid)
            if len(preview) < 10:
                preview.append({
                    'noteId': nid,
                    'cloze_prefix': value[:120]
                })

print({
    'dry_run': DRY_RUN,
    'checked_with_field': checked,
    'missing_field': missing_field,
    'notes_without_cloze_syntax': len(to_tag),
    'tag': TAG
})

print('Preview (first 10):')
for p in preview:
    print(p)

if not DRY_RUN and to_tag:
    for batch in chunked(to_tag, 500):
        invoke('addTags', notes=batch, tags=TAG)
    print(f"Tagged {len(to_tag)} notes with '{TAG}'")
elif not DRY_RUN:
    print('No notes needed tagging.')
else:
    print('DRY_RUN=True, no tags written.')

Found notes: 4571
{'dry_run': False, 'checked_with_field': 4571, 'missing_field': 0, 'notes_without_cloze_syntax': 185, 'tag': 'no_cloze_syntax'}
Preview (first 10):
{'noteId': 1734510465583, 'cloze_prefix': '飛行中の突然の 気流の変化に備えまして'}
{'noteId': 1734681071371, 'cloze_prefix': '\u200eあっ ここはValhallaのディレイを \u200e重ね掛けしてるかな'}
{'noteId': 1734681071416, 'cloze_prefix': '\u200eつまり習熟に \u200eもっていかなきゃいけない'}
{'noteId': 1734681071450, 'cloze_prefix': '\u200e手貸して いくよ グルグル回るよ \u200e（綴）一緒にいて！'}
{'noteId': 1739865122051, 'cloze_prefix': 'Subtitle'}
{'noteId': 1739865122055, 'cloze_prefix': 'どうぞ'}
{'noteId': 1739865122061, 'cloze_prefix': '\u200e（看護師）前職で 重いヘルニアを \u200e患ってらしたみたいで'}
{'noteId': 1739865122064, 'cloze_prefix': '\u200e野口さん 死んじゃったら \u200e困ります 俺'}
{'noteId': 1739865122065, 'cloze_prefix': '\u200e野口さん 死んじゃったら \u200e困ります 俺'}
{'noteId': 1739865122066, 'cloze_prefix': '\u200e全然 普通にコロコロ転がりました'}
Tagged 185 notes with 'no_cloze_syntax'


## Apply

1. Leave `DRY_RUN = True` and run all cells to confirm results.
2. Set `DRY_RUN = False` and run the processing cell again to write tags.

Make an Anki backup before applying changes.